In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive_output, FloatSlider, HBox, VBox, Layout, HTML, Output
from IPython.display import display

# ------------------------------------------------------------
# 1. DESCRIPTION
# ------------------------------------------------------------

description = HTML("""
<div style="
    border:1px solid #b9d7f5;
    border-radius:8px;
    padding:8px 10px;
    margin-bottom:10px;
    font-size:13px;
    line-height:1.35;
    background-color:#f7fbff;
">
<div><b>Purpose:</b> Explore the relation between the quality factor Q, the cutoff frequencies, and the bandwidth of a second-order band-pass filter.</div>
<div><b>What we see:</b> The normalized gain G(ω), the center frequency ω₀, the lower and upper cutoff frequencies ωc1 and ωc2, and the bandwidth Δω = ωc2 - ωc1.</div>
<div><b>What happens as we interact:</b> Increasing Q narrows the passband around ω₀, while decreasing Q widens it. Changing ω₀ moves the entire band along the frequency axis.</div>
</div>
""")

# ------------------------------------------------------------
# 2. CONTROLS
# ------------------------------------------------------------

q_slider = FloatSlider(min=0.50, max=5.00, step=0.05, value=1.00, description='Q:', continuous_update=True, readout=True, readout_format='.2f', style={'description_width':'30px'}, layout=Layout(width='220px'))

omega0_slider = FloatSlider(min=1.00, max=4.00, step=0.10, value=3.00, description='ω₀:', continuous_update=True, readout=True, readout_format='.2f', style={'description_width':'30px'}, layout=Layout(width='220px'))

# ------------------------------------------------------------
# 3. CUSTOM LEGEND
# ------------------------------------------------------------

legend_html = HTML("""
<div style="
    border:1px solid #cccccc;
    border-radius:5px;
    padding:7px 9px;
    width:125px;
    font-size:13px;
    line-height:1.7;
    background:white;
">

<div>
<span style="display:inline-block; width:32px; border-top:3px solid red; vertical-align:middle; margin-right:7px;"></span>
G(ω)
</div>

<div>
<span style="display:inline-block; width:32px; border-top:2px dashed #888888; vertical-align:middle; margin-right:7px;"></span>
1/√2
</div>

<div>
<span style="display:inline-block; width:32px; border-top:2px dashed black; vertical-align:middle; margin-right:7px;"></span>
ωc1
</div>

<div>
<span style="display:inline-block; width:32px; border-top:2px dotted black; vertical-align:middle; margin-right:7px;"></span>
ω₀
</div>

<div>
<span style="display:inline-block; width:32px; border-top:2px dash-dot #888888; vertical-align:middle; margin-right:7px;"></span>
ωc2
</div>

</div>
""")

parameter_label = HTML("<div style='font-size:14px; font-weight:bold; margin-top:12px; margin-bottom:4px;'>Parameters:</div>")

# ------------------------------------------------------------
# 4. OUTPUT WIDGETS
# ------------------------------------------------------------

plot_output = Output(layout=Layout(width='calc(100% - 230px)', overflow='visible'))

info_output = Output(layout=Layout(width='auto', overflow='visible'))

# ------------------------------------------------------------
# 5. MAIN PLOT FUNCTION
# ------------------------------------------------------------

def plot_bandpass(Q, omega0):

    omega = np.linspace(0.001, 10.0, 4000)

    r = omega / omega0

    denominator = np.sqrt((1.0 - r**2)**2 + (r / Q)**2)

    G = (r / Q) / denominator

    omega_c1 = (omega0 / (2.0 * Q)) * (np.sqrt(1.0 + 4.0 * Q**2) - 1.0)

    omega_c2 = (omega0 / (2.0 * Q)) * (np.sqrt(1.0 + 4.0 * Q**2) + 1.0)

    bandwidth = omega_c2 - omega_c1

    product = omega_c1 * omega_c2

    omega0_squared = omega0**2

    q_check = omega0 / bandwidth

    # --------------------------------------------------------
    # 6. PLOT
    # --------------------------------------------------------

    with plot_output:

        plot_output.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8.2, 4.6))

        ax.plot(omega, G, 'r-', linewidth=2.0)

        ax.axhline(1.0 / np.sqrt(2.0), color='gray', linestyle='--', linewidth=1.2)

        ax.axvline(omega_c1, color='black', linestyle='--', linewidth=1.5)

        ax.axvline(omega0, color='black', linestyle=':', linewidth=1.5)

        ax.axvline(omega_c2, color='gray', linestyle='-.', linewidth=1.5)

        ax.plot(omega_c1, 1.0 / np.sqrt(2.0), 'ko', markersize=4)

        ax.plot(omega0, 1.0, 'ko', markersize=4)

        ax.plot(omega_c2, 1.0 / np.sqrt(2.0), 'ko', markersize=4)

        arrow_y = 0.18

        ax.annotate('', xy=(omega_c2, arrow_y), xytext=(omega_c1, arrow_y), arrowprops=dict(arrowstyle='<->', linewidth=1.3))

        ax.text((omega_c1 + omega_c2) / 2.0, arrow_y + 0.07, 'Δω', horizontalalignment='center', fontsize=11)

        ax.set_xlim(0.0, 10.0)

        ax.set_ylim(0.0, 1.25)

        ax.set_xlabel('Angular Frequency ω  (rad/s)', fontsize=11)

        ax.set_ylabel('Normalized Gain G(ω)', fontsize=11)

        ax.set_title('Second-Order Band-Pass Filter', fontsize=13, fontweight='bold', pad=8)

        ax.grid(True, linestyle=':', alpha=0.35)

        plt.tight_layout()

        plt.show()

        plt.close(fig)

    # --------------------------------------------------------
    # 7. INFORMATION FRAME
    # --------------------------------------------------------

    info_html = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:7px 11px;
        font-size:12.5px;
        background:white;
        width:fit-content;
    ">

        <div style="
            display:flex;
            flex-direction:row;
            align-items:center;
            justify-content:flex-start;
            gap:20px;
            white-space:nowrap;
        ">

            <div>
                <b>Q:</b>
                <span style="color:#0066cc;">{Q:.2f}</span>
            </div>

            <div>
                <b>ω₀:</b>
                <span style="color:#0066cc;">{omega0:.2f} rad/s</span>
            </div>

            <div>
                <b>ωc1:</b>
                <span style="color:#0066cc;">{omega_c1:.2f} rad/s</span>
            </div>

            <div>
                <b>ωc2:</b>
                <span style="color:#0066cc;">{omega_c2:.2f} rad/s</span>
            </div>

            <div>
                <b>Δω:</b>
                <span style="color:#0066cc;">{bandwidth:.2f} rad/s</span>
            </div>

        </div>

        <div style="
            margin-top:5px;
            padding-top:5px;
            border-top:1px solid #eeeeee;
            white-space:nowrap;
        ">

            <b>Checks:</b>

            <span style="margin-left:12px;">
                ωc1 × ωc2 =
                <span style="color:#0066cc;">{product:.3f}</span>
            </span>

            <span style="margin-left:18px;">
                ω₀² =
                <span style="color:#0066cc;">{omega0_squared:.3f}</span>
            </span>

            <span style="margin-left:18px;">
                Q = ω₀ / Δω =
                <span style="color:#0066cc;">{q_check:.3f}</span>
            </span>

        </div>

    </div>
    """

    with info_output:

        info_output.clear_output(wait=True)

        display(HTML(info_html))

# ------------------------------------------------------------
# 8. INTERACTION
# ------------------------------------------------------------

interactive_controls = interactive_output(plot_bandpass, {'Q': q_slider, 'omega0': omega0_slider})

interactive_controls.layout.display = 'none'

# ------------------------------------------------------------
# 9. LEFT CONTROL COLUMN
# ------------------------------------------------------------

control_column = VBox([legend_html, parameter_label, q_slider, omega0_slider], layout=Layout(width='230px', min_width='230px', align_items='flex-start', padding='0px 0px 0px 4px'))

# ------------------------------------------------------------
# 10. MAIN AREA
# ------------------------------------------------------------

main_area = HBox([control_column, plot_output], layout=Layout(width='100%', align_items='flex-start', justify_content='flex-start', overflow='visible'))

# ------------------------------------------------------------
# 11. INFORMATION AREA
# ------------------------------------------------------------

info_spacer = HTML("", layout=Layout(width='230px', min_width='230px'))

info_area = HBox([info_spacer, info_output], layout=Layout(width='100%', align_items='flex-start', justify_content='flex-start', overflow='visible'))

# ------------------------------------------------------------
# 12. FINAL DISPLAY
# ------------------------------------------------------------

display(description)

display(main_area)

display(info_area)

display(interactive_controls)